In [1]:
# ============================================================
# ICUAdmissions.csv - NLP + K-Means
# ============================================================

# 1. Import libraries
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

# 2. Load dataset
df = pd.read_csv("ICUAdmissions.csv")

print("Dataset shape:", df.shape)
display(df.head())

# 3. Find text columns
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Text columns:", text_cols)

# Combine text columns
df["text"] = df[text_cols].fillna("").astype(str).agg(" ".join, axis=1)

# 4. NLP preprocessing
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return " ".join(words)

df["clean_text"] = df["text"].apply(clean_text)

# 5. TF-IDF
tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2
)

X = tfidf.fit_transform(df["clean_text"])

print("TF-IDF shape:", X.shape)

# 6. Find best K using silhouette score
scores = []

for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    scores.append(silhouette_score(X, labels))

best_k = range(2, 9)[np.argmax(scores)]

print("Best K:", best_k)
print("Best Silhouette Score:", max(scores))

# 7. Plot silhouette scores
plt.plot(range(2, 9), scores, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("Choosing K")
plt.show()

# 8. Final K-Means
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X)

# 9. Cluster sizes
print("\nCluster sizes:")
print(df["Cluster"].value_counts().sort_index())

# 10. Important words in each cluster
terms = tfidf.get_feature_names_out()

for i in range(best_k):
    indices = kmeans.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i} keywords:")
    print(", ".join(terms[indices]))

# 11. Visualise clusters
svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X)

plt.figure(figsize=(8, 6))
plt.scatter(
    X_2d[:, 0],
    X_2d[:, 1],
    c=df["Cluster"],
    cmap="viridis"
)

plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("ICU Admissions - K-Means Clusters")
plt.colorbar(label="Cluster")
plt.show()

# 12. Display results
display(df[["text", "Cluster"]].head(20))

# 13. Save results
df.to_csv("ICUAdmissions_KMeans_Results.csv", index=False)

print("\nDone! Results saved to ICUAdmissions_KMeans_Results.csv")

Dataset shape: (200, 21)


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,ID,Status,Age,Sex,Race,Service,Cancer,Renal,Infection,CPR,...,HeartRate,Previous,Type,Fracture,PO2,PH,PCO2,Bicarbonate,Creatinine,Consciousness
0,8,0,27,1,1,0,0,0,1,0,...,88,0,1,0,0,0,0,0,0,1
1,12,0,59,0,1,0,0,0,0,0,...,80,1,1,0,0,0,0,0,0,1
2,14,0,77,0,1,1,0,0,0,0,...,70,0,0,0,0,0,0,0,0,1
3,28,0,54,0,1,0,0,0,1,0,...,103,0,1,1,0,0,0,0,0,1
4,32,0,87,1,1,1,0,0,1,0,...,154,1,1,0,0,0,0,0,0,1


Text columns: []


ValueError: empty vocabulary; perhaps the documents only contain stop words